In [1]:
import sys
import numpy as np
import tensorflow as tf
import tensorflow.keras as K
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

2026-01-29 00:30:31.691759: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-29 00:30:31.701322: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-29 00:30:31.712006: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-29 00:30:31.715201: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-29 00:30:31.723771: I tensorflow/core/platform/cpu_feature_guar

In [2]:
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

In [3]:
layers = [4096, 2048, 1]
epochs = 50
act_func = tf.nn.relu
dropout = 0.5
input_dropout = 0.2
eta = 1e-5
norm = 'tanh'

In [4]:
X_tr, X_val, _, _, y_tr, y_val, _, _ = load(norm=norm)
print("Training data shape:", X_tr.shape)
print("Validation data shape:", X_val.shape)
print("NaN in X_tr:", np.isnan(X_tr).any())
print("NaN in y_tr:", np.isnan(y_tr).any())
print("Inf in X_tr:", np.isinf(X_tr).any())
print("Inf in y_tr:", np.isinf(y_tr).any())

Training data shape: (13884, 7063)
Validation data shape: (4614, 7063)
NaN in X_tr: False
NaN in y_tr: False
Inf in X_tr: False
Inf in y_tr: False


In [5]:
model = Sequential()
for i in range(len(layers)):
    if i == 0:
        model.add(Dense(
            layers[i],
            input_shape=(X_tr.shape[1],),
            activation=act_func,
            kernel_initializer='he_normal'
        ))
        model.add(Dropout(float(input_dropout)))
    elif i == len(layers) - 1:
        model.add(Dense(
            layers[i],
            activation='linear',
            kernel_initializer="he_normal"
        ))
    else:
        model.add(Dense(
            layers[i],
            activation=act_func,
            kernel_initializer="he_normal"
        ))
        model.add(Dropout(float(dropout)))

 00:30:39.884718: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2112] Could not identify NUMA node of platform GPU id 0, defaulting to 0.  Your kernel may not have been built with NUMA support.
I0000 00:00:1769646639.884751    3567 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-29 00:30:39.884770: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 26831 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5090, pci bus id: 0000:02:00.0, compute capability: 12.0


In [6]:
model.compile(
    loss='mean_squared_error',
    optimizer=K.optimizers.SGD(
        learning_rate=float(eta),
        momentum=0.5
    )
)
model.summary()

_________________________________________________________________


In [7]:
hist = model.fit(
    X_tr, y_tr,
    epochs=epochs,
    batch_size=64,
    shuffle=True,
    validation_data=(X_val, y_val),
    verbose=1   
)

217/217 [==============================] - 1s 3ms/step - loss: 91.9796 - val_loss: 87.1200


In [8]:
val_loss = hist.history['val_loss']
train_loss = hist.history['loss']
print("Final training loss:", train_loss[-1])
print("Final validation loss:", val_loss[-1])
model.reset_states()

Final training loss: 91.97958374023438
Final validation loss: 87.11997985839844
